## SCIAC Softball Web Scraping

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [2]:
def scrape_team(team_name, team_url, year):
    url = f"https://{team_url}.com/sports/softball/schedule/{year}"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    games = soup.find_all("div", class_="sidearm-schedule-game-row")

    data = []

    for g in games:
        opponent = g.find("div", class_="sidearm-schedule-game-opponent-name")
        date = g.find("div", class_="sidearm-schedule-game-opponent-date")
        result = g.find("div", class_="sidearm-schedule-game-result")

        opponent_text = opponent.get_text(strip=True) if opponent else None
        date_text = date.get_text(" ", strip=True) if date else None

        outcome = None
        score = None
        score_for = None
        score_against = None

        if result:
            spans = result.find_all("span")
            if len(spans) >= 3:
                outcome = spans[1].text.strip().replace(",", "")
                score = spans[2].text.strip()

                # score parsing
                if score and "-" in score:
                    nums = re.findall(r'\d+', score)
                    if len(nums) == 2:
                        score_for = int(nums[0])
                        score_against = int(nums[1])

        data.append({
            "team": team_name,
            "year": year,
            "opponent": opponent_text,
            "date": date_text,
            "outcome": outcome,
            "score": score,
            "score_for": score_for,
            "score_against": score_against
        })

    return pd.DataFrame(data)

In [3]:
df_test = scrape_team("California Lutheran", "clusports", 2019)
df_test.head()

,team,year,opponent,date,outcome,score,score_for,score_against
0,California Lutheran,2019,La Verne,Feb 16 (Sat) 12 pm,L,4-11,4,11
1,California Lutheran,2019,La Verne,Feb 16 (Sat) 2 pm,L,4-6,4,6
2,California Lutheran,2019,Claremont-Mudd-Scripps,Feb 23 (Sat) 12 pm,L,6-9,6,9
3,California Lutheran,2019,Claremont-Mudd-Scripps,Feb 23 (Sat) 2 pm,L,0-10,0,10
4,California Lutheran,2019,Hiram,Mar 4 (Mon) 12 pm,W,5-1,5,1


In [4]:
years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

df_clu = pd.concat(
    [scrape_team("California Lutheran", "clusports", y) for y in years],
    ignore_index=True
)

df_clu.head()

,team,year,opponent,date,outcome,score,score_for,score_against
0,California Lutheran,2019,La Verne,Feb 16 (Sat) 12 pm,L,4-11,4.0,11.0
1,California Lutheran,2019,La Verne,Feb 16 (Sat) 2 pm,L,4-6,4.0,6.0
2,California Lutheran,2019,Claremont-Mudd-Scripps,Feb 23 (Sat) 12 pm,L,6-9,6.0,9.0
3,California Lutheran,2019,Claremont-Mudd-Scripps,Feb 23 (Sat) 2 pm,L,0-10,0.0,10.0
4,California Lutheran,2019,Hiram,Mar 4 (Mon) 12 pm,W,5-1,5.0,1.0


In [5]:
teams = [
    ("California Lutheran", "clusports"),
    ("Redlands", "goredlands"),
    ("Chapman", "chapmanathletics"),
    ("Pomona-Pitzer", "sagehens"),
    ("Whittier", "wcpoets"),
    ("Occidental", "oxyathletics"),
    ("La Verne", "leopardathletics")
]

In [6]:
dfs = []

for team_name, team_url in teams:
    for year in years:
        df_team = scrape_team(team_name, team_url, year)
        dfs.append(df_team)

df_final = pd.concat(dfs, ignore_index=True)

df_final.shape

(2225, 8)

In [7]:
df_final.head()

,team,year,opponent,date,outcome,score,score_for,score_against
0,California Lutheran,2019,La Verne,Feb 16 (Sat) 12 pm,L,4-11,4.0,11.0
1,California Lutheran,2019,La Verne,Feb 16 (Sat) 2 pm,L,4-6,4.0,6.0
2,California Lutheran,2019,Claremont-Mudd-Scripps,Feb 23 (Sat) 12 pm,L,6-9,6.0,9.0
3,California Lutheran,2019,Claremont-Mudd-Scripps,Feb 23 (Sat) 2 pm,L,0-10,0.0,10.0
4,California Lutheran,2019,Hiram,Mar 4 (Mon) 12 pm,W,5-1,5.0,1.0


### CMS Note

We were not able to include CMS in this data set. This is because the schedule is loaded dynamically using JavaScript after the page loads. Since requests does not execute JavaScript, the game data was not present in the HTML response.

In [8]:
df_cms_test = scrape_team("CMS", "cmsathletics", 2024)
df_cms_test.head()

""


## Cleaning

### Clean Date Column

In [9]:
def clean_date(date_str, year):
    if pd.isna(date_str):
        return None

    # remove (Sat), (Fri), etc.
    date_str = re.sub(r'\([^)]*\)', '', date_str)

    # standardize AM/PM
    date_str = date_str.replace('am', 'AM').replace('pm', 'PM')

    return f"{date_str.strip()} {year}"

In [10]:
df_final['date'] = df_final.apply(
    lambda row: clean_date(row['date'], row['year']),
    axis=1
)

df_final['date'] = pd.to_datetime(df_final['date'], errors='coerce')

df_final[['date']].head()

,date
0,2019-02-16 12:00:00
1,2019-02-16 14:00:00
2,2019-02-23 12:00:00
3,2019-02-23 14:00:00
4,2019-03-04 12:00:00


### Opponent Cleaning

In [38]:
def clean_matchup_text(name):
    if not isinstance(name, str):
        return name

    # remove anything after "vs"
    name = re.split(r'\bvs\b', name)[0]

    # remove numbers (rankings, seeds)
    name = re.sub(r'\d+', '', name)

    # remove words like "seed"
    name = name.replace("seed", "")

    return name.strip()

In [39]:
def clean_basic_text(name):
    if not isinstance(name, str):
        return name

    name = name.lower()

    # remove parentheses (DH, etc.)
    name = re.sub(r'\([^)]*\)', '', name)

    # remove punctuation except hyphen
    name = re.sub(r'[^\w\s-]', '', name)

    # fix spacing
    name = re.sub(r'\s+', ' ', name).strip()

    return name

In [40]:
def remove_school_words(name):
    if not isinstance(name, str):
        return name

    words_to_remove = ["university", "college"]

    for word in words_to_remove:
        name = name.replace(word, "")

    name = re.sub(r'\s+', ' ', name).strip()

    return name

In [41]:
# Standardize SCIAC Teams
def standardize_sciac(name):
    if not isinstance(name, str):
        return name

    if "pomona" in name or "pitzer" in name:
        return "pomona-pitzer"

    if "cal lutheran" in name or "california lutheran" in name:
        return "california-lutheran"

    if "redlands" in name:
        return "redlands"

    if "occidental" in name:
        return "occidental"

    if "la verne" in name:
        return "la-verne"

    if "whittier" in name:
        return "whittier"

    if "chapman" in name:
        return "chapman"

    if "claremont" in name or "cms" in name:
        return "claremont-mudd-scripps"

    return name

In [42]:
df_final["opponent"] = (
    df_final["opponent"]
    .apply(clean_matchup_text)
    .apply(clean_basic_text)
    .apply(remove_school_words)
    .apply(standardize_sciac)
)

In [43]:
df_final["opponent"].value_counts().head(15)

,count
opponent,
chapman,138
claremont-mudd-scripps,134
redlands,133
california-lutheran,122
occidental,120
la-verne,114
pomona-pitzer,110
whittier,102
williams,46


### Clean Team Column

In [30]:
def clean_team_name(name):
    if not isinstance(name, str):
        return name

    name = name.lower()

    # remove extra words
    name = name.replace("university", "")
    name = name.replace("college", "")

    # fix spacing
    name = name.strip()

    # standardize SCIAC teams
    if "california lutheran" in name:
        return "california-lutheran"
    if "la verne" in name:
        return "la-verne"
    if "pomona" in name or "pitzer" in name:
        return "pomona-pitzer"
    if "claremont" in name:
        return "claremont-mudd-scripps"

    return name

In [31]:
df_final["team"] = df_final["team"].apply(clean_team_name)

## Feature Engineer

In [32]:
# score difference
df_final["score_diff"] = df_final["score_for"] - df_final["score_against"]

# total points in game
df_final["total_points"] = df_final["score_for"] + df_final["score_against"]

# win indicator (True/False)
df_final["win"] = df_final["outcome"] == "W"

# margin category
def margin_type(diff):
    if pd.isna(diff):
        return None
    elif abs(diff) <= 2:
        return "close"
    elif abs(diff) >= 5:
        return "blowout"
    else:
        return "moderate"

df_final["margin_type"] = df_final["score_diff"].apply(margin_type)

## Final Cleaned Data Set

In [33]:
df_final

,team,year,opponent,date,outcome,score,score_for,score_against,month,score_diff,total_points,win,margin_type
0,california-lutheran,2019,la-verne,2019-02-16 12:00:00,L,4-11,4.0,11.0,2.0,-7.0,15.0,False,blowout
1,california-lutheran,2019,la-verne,2019-02-16 14:00:00,L,4-6,4.0,6.0,2.0,-2.0,10.0,False,close
2,california-lutheran,2019,claremont-mudd-scripps,2019-02-23 12:00:00,L,6-9,6.0,9.0,2.0,-3.0,15.0,False,moderate
3,california-lutheran,2019,claremont-mudd-scripps,2019-02-23 14:00:00,L,0-10,0.0,10.0,2.0,-10.0,10.0,False,blowout
4,california-lutheran,2019,hiram,2019-03-04 12:00:00,W,5-1,5.0,1.0,3.0,4.0,6.0,True,moderate
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2220,la-verne,2025,redlands,2025-04-26 12:00:00,L,3-11,3.0,11.0,4.0,-8.0,14.0,False,blowout
2221,la-verne,2025,redlands,2025-04-26 14:00:00,L,1-6,1.0,6.0,4.0,-5.0,7.0,False,blowout
2222,la-verne,2025,whittier,2025-05-02 15:00:00,W,3-0,3.0,0.0,5.0,3.0,3.0,True,moderate
2223,la-verne,2025,whittier,2025-05-03 12:00:00,L,0-8,0.0,8.0,5.0,-8.0,8.0,False,blowout


## Final Observations

In [34]:
# Win percentages by team
df_final["win_flag"] = df_final["outcome"].apply(lambda x: 1 if x == "W" else 0)

df_final.groupby("team")["win_flag"].mean().sort_values(ascending=False)

,win_flag
team,
redlands,0.542208
chapman,0.522648
pomona-pitzer,0.460208
whittier,0.421769
la-verne,0.413127
california-lutheran,0.325000
occidental,0.175182


In [35]:
# Average points scored vs allowed
df_final.groupby("team")[["score_for", "score_against"]].mean()

,score_for,score_against
team,,
california-lutheran,4.350962,6.052885
chapman,4.598425,3.893701
la-verne,4.545455,4.522727
occidental,3.269912,6.632743
pomona-pitzer,5.040984,4.077869
redlands,5.864341,4.189922
whittier,5.188285,4.866109


In [36]:
# Win % by team per year
df_final.groupby(["team", "year"])["win_flag"].mean()

team                 year
california-lutheran  2019    0.300000
                     2020    0.161290
                     2021    0.166667
                     2022    0.282051
                     2023    0.333333
                     2024    0.416667
                     2025    0.529412
chapman              2019    0.441860
                     2020    0.153846
                     2021    0.434783
                     2022    0.651163
                     2023    0.600000
                     2024    0.586957
                     2025    0.687500
la-verne             2019    0.395349
                     2020    0.250000
                     2021    0.300000
                     2022    0.604651
                     2023    0.538462
                     2024    0.317073
                     2025    0.410256
occidental           2019    0.125000
                     2020    0.027778
                     2021    0.095238
                     2022    0.275000
                     2023    0.230769
                     2024    0.194444
                     2025    0.268293
pomona-pitzer        2019    0.704545
                     2020    0.121212
                     2021    0.279070
                     2022    0.488372
                     2023    0.489362
                     2024    0.461538
                     2025    0.600000
redlands             2019    0.372093
                     2020    0.305556
                     2021    0.608696
                     2022    0.541667
                     2023    0.574468
                     2024    0.600000
                     2025    0.714286
whittier             2019    0.480000
                     2020    0.142857
                     2021    0.357143
                     2022    0.550000
                     2023    0.500000
                     2024    0.461538
                     2025    0.463415
Name: win_flag, dtype: float64

## Save to SQL

In [44]:
import sqlite3

# create database
conn = sqlite3.connect("softball_data.db")

# save dataframe to SQL table
df_final.to_sql("games", conn, if_exists="replace", index=False)

# close connection
conn.close()

In [45]:
from google.colab import files
files.download("softball_data.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Data Source / Who Is Interested / Challenges

I took data from the SCIAC softball conference for the years 2019-2025. This data includes the game results, scores, and opponent information. I then parsed through it and tried to pull some interesting statistics that I thought were meaningful.

I believe coaches and players would find this work interesting. If I was able to pull hitting/pitching statistics from each game I think this would be really helpful to analysts. I also think the SCIAC as an organization would think this is cool. This could be applied to many different teams not just softball in the SCIAC!

I definitely had some challenges with this project. The first being that I could not scrape from CMS website. After attempting to fix that, and failing, I went on to cleaning the data. This was also very challenging. The datetimes were actually a it easier to clean than the opponent names. The names took me a long time to standardize and get put in the correct category. I had lots of overlap like with PLU and CLU but I ended up correcting it.
